<a href="https://colab.research.google.com/github/toche7/BasicAIAgentColab/blob/main/BasicHarness1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent: Connect LLM

### ขั้นตอนที่ 1: การกำหนดค่าพื้นฐานและเตรียมการเรียกใช้งาน API
ในส่วนนี้เราจะนำเข้าไลบรารีที่จำเป็น ได้แก่:
- `os` และ `sys` สำหรับการจัดการระบบ
- `httpx` ซึ่งเป็น HTTP client ที่ใช้ในการส่งคำร้องขอ (Request) ไปยังปลายทาง
- `userdata` จาก `google.colab` เพื่อดึงค่า API Key ที่เก็บไว้อย่างปลอดภัย

นอกจากนี้จะมีการกำหนด URL ปลายทางของ Groq API และโมเดลที่ต้องการเลือกใช้งาน

In [1]:
"""Step 1: one file. Send a conversation to an OpenAI-compatible API (Groq) and print the reply."""
import os    # This is a standard Python library used to interact with the operating system.
import sys    # This standard library provides access to variables and functions that interact closely with the Python interpreter.
import httpx  # This is a modern, powerful third-party HTTP client for Python.
from google.colab import userdata

URL = "https://api.groq.com/openai/v1/chat/completions"
MODEL = "qwen/qwen3.8-27b"

### ขั้นตอนที่ 2: การดึง API Key และฟังก์ชันการสนทนา (Chat Function)
โค้ดส่วนนี้ทำหน้าที่:
1. ดึงข้อมูลรหัสลับ `GROQ_API_KEY` จาก Colab Userdata หรือ Environment Variable
2. นิยามฟังก์ชัน `chat` เพื่อรับประวัติบทสนทนา (`messages`) แล้วสร้าง Payload ส่งแบบ POST request ไปยัง Groq API และคืนค่าผลลัพธ์ข้อความที่ตอบกลับมาจากโมเดล

In [2]:
try:
        api_key = userdata.get('GROQ_API_KEY')
except Exception:
        api_key = os.environ.get('GROQ_API_KEY')


def chat(messages: list[dict]) -> dict:
    """POST the conversation with tools configuration to Groq API."""
    if not api_key:
        raise ValueError("GROQ_API_KEY not found.")

    body = {"model": MODEL, "messages": messages, "max_tokens": 1000}

    headers = {"Authorization": f"Bearer {api_key}"}
    resp = httpx.post(URL, headers=headers, json=body, timeout=60)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]

### ขั้นตอนที่ 3: การเริ่มต้นรันโปรแกรมและส่งคำถาม
ในบล็อกนี้เราจะกำหนดคำถามเริ่มต้น เช่น *"What is an AI agent?"* และทำการเรียกใช้ฟังก์ชัน `chat` ที่สร้างไว้ก่อนหน้า จากนั้นจะแสดงผลลัพธ์คำตอบที่ได้รับกลับมาจากการประมวลผลของ LLM บนระบบของ Groq

In [21]:

if __name__ == "__main__":
    # In Colab, sys.argv includes notebook runtime arguments, so we override or fallback to a default prompt
    question = "Where is Krabi, in 10 words?"
    messages = [{"role": "user", "content": question}]
    reply = chat(messages)
    print(reply['content'])


Southern Thailand, bordering Myanmar, featuring stunning limestone karsts and beaches.


In [22]:
reply

{'role': 'assistant',
 'content': 'Southern Thailand, bordering Myanmar, featuring stunning limestone karsts and beaches.'}

In [25]:
reply.keys()

dict_keys(['role', 'content'])

In [26]:
reply['role']

'assistant'

In [27]:
reply['content']

'Southern Thailand, bordering Myanmar, featuring stunning limestone karsts and beaches.'